# 01 · Exploration des données (EDA)

**Objectif** : comprendre ce qu'il y a dans `data/jobs.db` avant d'analyser quoi que ce soit.

On répond à 5 questions : combien d'offres ? quelles colonnes sont vides ? combien ont un salaire ? sur quelles dates ? les descriptions sont-elles complètes ?

In [ ]:
import sys
from pathlib import Path

# Racine du projet : fonctionne que le notebook soit lance depuis notebooks/ ou depuis la racine
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import common

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print("Projet :", ROOT)

## 1. Chargement
`only_relevant=False` pour voir **toutes** les offres, pertinentes ou non.

In [ ]:
jobs = common.load_jobs(only_relevant=False)
print(f"{len(jobs)} offres, {jobs.shape[1]} colonnes")
jobs.head()

## 2. Valeurs manquantes
Une colonne très vide est peu exploitable (souvent le salaire).

In [ ]:
missing = jobs.isna().mean().mul(100).round(1).sort_values(ascending=False)
missing = missing[missing > 0]
if missing.empty:
    print("Aucune valeur manquante.")
else:
    ax = missing.plot.barh(figsize=(8, max(2, 0.35 * len(missing))), color="#d9534f")
    ax.set_xlabel("% de valeurs manquantes")
    ax.invert_yaxis()
    plt.show()

## 3. Pertinence des offres
`pertinent` : 0 = hors sujet, 1 = titre OK, 2 = titre + description OK.

In [ ]:
if "pertinent" in jobs.columns:
    counts = jobs["pertinent"].value_counts().sort_index()
    display(counts.to_frame("offres"))
    counts.plot.bar(rot=0, color="#5b8def", figsize=(5, 3))
    plt.ylabel("Nombre d'offres"); plt.show()
else:
    print("Pas de colonne `pertinent`.")

## 4. Sources des offres

In [ ]:
source_col = next((c for c in jobs.columns if c.lower() in ("source", "origine", "provider")), None)
if source_col:
    display(jobs[source_col].value_counts().to_frame("offres"))
else:
    print("Pas de colonne source : ajoute-en une quand tu integres France Travail.")

## 5. Salaires
La colonne est détectée automatiquement (`common.detect_salary`). Les valeurs hors de 800–20 000 €/mois sont ignorées.

In [ ]:
sal = jobs["salaire_m"].dropna()
print(f"Offres avec salaire : {len(sal)} / {len(jobs)} ({len(sal) / len(jobs) * 100:.0f} %)")
if len(sal):
    print(sal.describe().round(0).to_string())
    sal.plot.hist(bins=30, figsize=(7, 3.5), color="#5cb85c")
    plt.xlabel("Salaire mensuel (EUR)"); plt.show()

## 6. Dates
Sans plusieurs dates différentes, impossible de mesurer une tendance.

In [ ]:
dates = jobs["date_offre"].dropna()
if dates.empty:
    print("Aucune date exploitable : voir DATE_COL dans common.py")
else:
    print(f"Du {dates.min():%Y-%m-%d} au {dates.max():%Y-%m-%d} ({(dates.max() - dates.min()).days} jours)")
    dates.dt.to_period("W").value_counts().sort_index().plot(kind="bar", figsize=(8, 3))
    plt.ylabel("Offres par semaine"); plt.show()

## 7. Longueur des descriptions
Si beaucoup de descriptions s'arrêtent à la même longueur (ex. 500 caractères), elles sont **tronquées** : on rate des compétences.

In [ ]:
length = jobs["description"].fillna("").str.len()
print(length.describe().round(0).to_string())
top = length.value_counts().head(3)
print("\nLongueurs les plus frequentes :")
print(top.to_string())
length.plot.hist(bins=40, figsize=(7, 3.5), color="#f0ad4e")
plt.xlabel("Caracteres par description"); plt.show()

## À retenir
- Colonnes trop vides → à ne pas utiliser, ou à compléter avec France Travail.
- Part d'offres avec salaire < 30 % → présenter les salaires avec prudence.
- Une seule date → relancer la collecte régulièrement.
- Descriptions tronquées → intégrer France Travail pour de meilleures descriptions.